## Purpose

Filter and calculate annual and monthly means of monthly mixed layer depth, temperature, X and Y seawater velocities from MOM6 for yellowtail.

* For the annual means, each variable is filtered for latitudes, longitudes (based on bathymetry), and months that are relevant to that life stage.
* For the monthly means, each variables if filtered for latitudes and longitudes (based on bathymetry) that are relevant to a particular life stage. Means are calculated for every month.

## Set up

In [22]:
import xarray as xr
import numpy as np
import pandas as pd
import time

# Adding this module to allow Earthmover Arraylake access - RHS 6/8/2026
from arraylake import Client

## Download data

Script assumes you have these files in a `OcModels/Data/MOM6` folder. To download necessary inputs, run:

*Depth (2-d)*
* *deptho* `wget -O ocean_static.deptho.nc https://psl.noaa.gov/thredds/fileServer/Projects/CEFI/regional_mom6/cefi_derivative/northeast_pacific/full_domain/hindcast/monthly/regrid/r20250509/static/ocean_static.deptho.nc`


**You should not have to download the following files. You should be able to access the data directly from Earthmover.**

*Mixed layer depth (2-d)*
* *MLD_003* `wget -O MLD_003.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc https://psl.noaa.gov/thredds/fileServer/Projects/CEFI/regional_mom6/cefi_portal/northeast_pacific/full_domain/hindcast/monthly/regrid/r20250509/MLD_003.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc`

*Temperature (3-d)*
* *thetao* `wget -O thetao.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc https://psl.noaa.gov/thredds/fileServer/Projects/CEFI/regional_mom6/cefi_portal/northeast_pacific/full_domain/hindcast/monthly/regrid/r20250509/thetao.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc`

*X & Y sea water velocities (3-d)*
* *uo_rotate* `wget -O uo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc https://psl.noaa.gov/thredds/fileServer/Projects/CEFI/regional_mom6/cefi_portal/northeast_pacific/full_domain/hindcast/monthly/regrid/r20250509/uo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc`
* *vo_rotate* `wget -O vo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc https://psl.noaa.gov/thredds/fileServer/Projects/CEFI/regional_mom6/cefi_portal/northeast_pacific/full_domain/hindcast/monthly/regrid/r20250509/vo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc`

Depth was based on a link Chia-Wei shared with me. The remaining queries were generated on the [CEFI Data Portal](https://psl.noaa.gov/cefi_portal/#data_access).
* Region: NEP
* Subregion: full domain
* Experiment Type: hindcast
* Output Frequency: monthly regrid
* Release: r20250509
* Data Category: ocean_monthly_z (for 3-d) or ocean_monthly (for 2-d)

In the future, it'd be nice to do this on the raw data rather than the gridded data.

# Annual Means

## Life stage dictionary

Based on [Data Availability](https://docs.google.com/document/d/1P8D0kH2xn4NYBc0ib3rYfSO0KAiyDNQ_kZgG4Qub7qY/edit?tab=t.0#heading=h.z5x77oqxpj4i), define filters for each lifestage for time, bottom depth, and latitude.

For northern yellowtail, the bottom depth is used to determine the longitudinal extent, and the average is taken across the whole water column. For example, if the longitudinal extent is bottom depths 90-180m, the netCDF is filtered for isobaths where the depth is \[90, 180], and the average is calculated for all the water \[0, 180] at those depths.

In [23]:
# assign months, latitudes, and depths to filter
# all are integers, depths are in meters
# filters are inclusive (closed interval), e.g. depth 0-180 includes depths 0 and 180 and every depth between

lifestage_dict = {
    "cop": { # copulation
        "min_month": 8,
        "max_month": 10,
        "min_lat": 40,
        "max_lat": 48,
        "min_depth": 90,
        "max_depth": 180
    }, 
    "part": { # partuition
        "min_month": 1,
        "max_month": 4,
        "min_lat": 40,
        "max_lat": 48,
        "min_depth": 0,
        "max_depth": 180
    },
    "larv": { # larvae
        "min_month": 2,
        "max_month": 3,
        "min_lat": 40,
        "max_lat": 48,
        "min_depth": 0,
        "max_depth": 90
    },
    "pjuv": { # pelagic juvenile
        "min_month": 4,
        "max_month": 8,
        "min_lat": 40,
        "max_lat": 48,
        "min_depth": 30,
        "max_depth": 130
    }
}

## Load datasets

In [24]:
# depth is used to filter longitudinal extent
ds_depth = xr.open_dataset("../../Data/MOM6/input/ocean_static.deptho.nc")

# # mixed layer depth
# ds_mld = xr.open_dataset("../../Data/MOM6/input/MLD_003.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc")
#
# # temperature
# ds_temp = xr.open_dataset("../../Data/MOM6/input/thetao.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc")
#
# # X and Y sea water velocities
# ds_xv = xr.open_dataset("../../Data/MOM6/input/uo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc")
# ds_yv = xr.open_dataset("../../Data/MOM6/input/vo_rotate.nep.full.hcast.monthly.regrid.r20250509.199301-202412.nc")

client = Client()
repo = client.get_repo("NOAA-PMEL/cefi-nep-hindcast-monthly")
session = repo.readonly_session(branch="main")
aux = xr.open_zarr(session.store, zarr_format=3, group="regrid/aux1")

# The double brackets extract these as dataset to keep the same structure as the previous code.

ds_mld = aux[['MLD_003']]

main = xr.open_zarr(session.store, zarr_format=3, group="regrid/main")

ds_temp = main[['thetao']]
ds_xv = main[['uo_rotate']]
ds_yv = main[['vo_rotate']]

## Functions

In [25]:
'''
Print the size and dimensions of a dataset for a variable named 'var_name'
If nan_bool is true, also count the number of non-NA records (# and %) for that variable

Inputs:
- dataset: xarray Dataset with the variable var_name
- var_name: string with the variable name to count
- nan_bool: boolean, to print record count and % for the variable

Outputs:
- None, will print output
'''

def print_ds_info(dataset, var_name, nan_bool = False):
    denominator = dataset[var_name].size
    print(f'{denominator} records with dimensions {dataset[var_name].shape}')
    
    if nan_bool:
        numerator = np.isfinite(dataset[var_name].data).sum()
        percent = numerator / denominator * 100
        print(f'{numerator} records ({percent:.{2}f}%) are not np.nan')
        
    return None

In the future, it'd be nice to adjust these functions to have an argument to only run if the output file isn't in the directory already, so the notebook could run top to bototm and only re-caculate missing indices.

In [26]:
'''
Calculate annual mean for a 2-D variable

Inputs:
- var_dataset: xarray Dataset with the variable to average
- depth_dataset: xarray Dataset with depth to filter
- ds_var_name: string with the variable name, used to extract from var_dataset
- df_var_name: string with the variable name, used to rename column in returned dataframe
- filter_dict: dictionary with relevant filters
    - min_mon, max_mon: minimum and maximum month (integers, inclusive)
    - min_lat, max_lat: minimum and maximum latitude (integers or floats, inclusive)
    - min_depth, max_depth: minimum and maximum depth to filter the longitudinal extent
- print_counts: boolean, to print record count and % at each step; defaults to True for 2-D

Outputs:
- ds_df: pandas dataframe with an annual average of the variable
'''

def annual_mean_2d(var_dataset, depth_dataset, ds_var_name, df_var_name, filter_dict, print_counts = True):

    # start timer
    start_time = time.time()
    
    print("---" + df_var_name + "---")
    print_ds_info(var_dataset, ds_var_name, print_counts)
    
    # merge files
    ds_merged = xr.merge([var_dataset, depth_dataset])
    print_ds_info(ds_merged, ds_var_name, print_counts)

    # create list with months
    month_list = [x for x in range(filter_dict["min_month"], filter_dict["max_month"]+1)]
    # select times and latitudes
    ds_selected = ds_merged.sel(time =  ds_merged.time.dt.month.isin(month_list),
                                lat = slice(filter_dict["min_lat"], filter_dict["max_lat"]))
    print_ds_info(ds_selected, ds_var_name, print_counts)
    
    # mask depths
    ds_masked = ds_selected.where(
        (ds_selected.deptho >= filter_dict["min_depth"]) & 
        (ds_selected.deptho <= filter_dict["max_depth"])
    )
    print_ds_info(ds_masked, ds_var_name, print_counts)
    
    # calculate mean
    ds_mean = ds_masked[ds_var_name].groupby('time.year').mean(dim = ['time', 'lon', 'lat'])
    
    # create pandas dataframe
    ds_df = ds_mean.to_dataframe()
    
    # rename column from dataset to dataframe variable name
    ds_df.rename(columns = {ds_var_name:df_var_name}, inplace = True)

    # end timer and print runtime
    end_time = time.time()
    print(f'Runtime: {round((end_time - start_time) / 60, 2)} minutes')

    # write output
    ds_df.to_csv("../../Data/MOM6/interim/yellowtail_MOM6_" + df_var_name + ".csv")
    
    return ds_df

In [27]:
'''
Calculate annual mean for a 3-D variable

Inputs:
- var_dataset: xarray Dataset with the variable to average
- depth_dataset: xarray Dataset with depth to filter
- ds_var_name: string with the variable name, used to extract from var_dataset
- df_var_name: string with the variable name, used to rename column in returned dataframe
- filter_dict: dictionary with relevant filters
    - min_mon, max_mon: minimum and maximum month (integers, inclusive)
    - min_lat, max_lat: minimum and maximum latitude (integers or floats, inclusive)
    - min_depth, max_depth: minimum and maximum depth to filter the longitudinal extent
- print_counts: boolean, to print record count and % at each step; defaults to False for 3-D

Outputs:
- ds_df: pandas dataframe with an annual average of the variable
'''

def annual_mean_3d(var_dataset, depth_dataset, ds_var_name, df_var_name, filter_dict, print_counts = False):

    # start timer
    start_time = time.time()
    
    print("---" + df_var_name + "---")
    print_ds_info(var_dataset, ds_var_name, print_counts)
    
    # merge files
    ds_merged = xr.merge([var_dataset, depth_dataset])
    print_ds_info(ds_merged, ds_var_name, print_counts)

    # create list with months
    month_list = [x for x in range(filter_dict["min_month"], filter_dict["max_month"]+1)]
    # select times and latitudes
    ds_selected = ds_merged.sel(time =  ds_merged.time.dt.month.isin(month_list),
                                lat = slice(filter_dict["min_lat"], filter_dict["max_lat"]))
    print_ds_info(ds_selected, ds_var_name, print_counts)
    
    # mask depths
    ds_masked = ds_selected.where(
        (ds_selected.deptho >= filter_dict["min_depth"]) & 
        (ds_selected.deptho <= filter_dict["max_depth"])
    )
    print_ds_info(ds_masked, ds_var_name, print_counts)
    
    # calculate mean
    ds_mean = ds_masked[ds_var_name].groupby('time.year').mean(dim = ['time', 'z_l', 'lon', 'lat'])
    
    # create pandas dataframe
    ds_df = ds_mean.to_dataframe()
    
    # rename column from dataset to dataframe variable name
    ds_df.rename(columns = {ds_var_name:df_var_name}, inplace = True)

    # end timer and print runtime
    end_time = time.time()
    print(f'Runtime: {round((end_time - start_time) / 60, 2)} minutes')

    # write output
    ds_df.to_csv("../../Data/MOM6/interim/yellowtail_MOM6_" + df_var_name + ".csv")
    
    return ds_df

## Calculate means

In [28]:
# mixed layer depths
MLDpart = annual_mean_2d(ds_mld, ds_depth, "MLD_003", "MLDpart", lifestage_dict['part'], True)
MLDlarv = annual_mean_2d(ds_mld, ds_depth, "MLD_003", "MLDlarv", lifestage_dict['larv'], True)
MLDpjuv = annual_mean_2d(ds_mld, ds_depth, "MLD_003", "MLDpjuv", lifestage_dict['pjuv'], True)

---MLDpart---
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
4359344 records with dimensions (136, 94, 341)
1917056 records (43.98%) are not np.nan
4359344 records with dimensions (136, 94, 341)
13192 records (0.30%) are not np.nan
Runtime: 0.91 minutes
---MLDlarv---
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
2179672 records with dimensions (68, 94, 341)
958528 records (43.98%) are not np.nan
2179672 records with dimensions (68, 94, 341)
3128 records (0.14%) are not np.nan
Runtime: 0.0 minutes
---MLDpjuv---
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
110054340 records with dimensions (396, 815, 341)
39259440 records (35.67%) are not np.nan
5320964 records with dimensions

In [29]:
# temperatures
Tcop = annual_mean_3d(ds_temp, ds_depth, "thetao", "Tcop", lifestage_dict['cop'], False)
Tpart = annual_mean_3d(ds_temp, ds_depth, "thetao", "Tpart", lifestage_dict['part'], False)

---Tcop---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
160013568 records with dimensions (96, 52, 94, 341)
160013568 records with dimensions (96, 52, 94, 341)
Runtime: 5.61 minutes
---Tpart---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
220018656 records with dimensions (132, 52, 94, 341)
220018656 records with dimensions (132, 52, 94, 341)
Runtime: 5.11 minutes


In [30]:
# X sea water velocities
XVlarv = annual_mean_3d(ds_xv, ds_depth, "uo_rotate", "XVlarv", lifestage_dict['larv'], False)
XVpjuv = annual_mean_3d(ds_xv, ds_depth, "uo_rotate", "XVpjuv", lifestage_dict['pjuv'], False)

---XVlarv---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
110009328 records with dimensions (66, 52, 94, 341)
110009328 records with dimensions (66, 52, 94, 341)
Runtime: 6.21 minutes
---XVpjuv---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
271689704 records with dimensions (163, 52, 94, 341)
271689704 records with dimensions (163, 52, 94, 341)
Runtime: 6.2 minutes


In [31]:
# Y sea water velocities
YVlarv = annual_mean_3d(ds_yv, ds_depth, "vo_rotate", "YVlarv", lifestage_dict['larv'], False)
YVpjuv = annual_mean_3d(ds_yv, ds_depth, "vo_rotate", "YVpjuv", lifestage_dict['pjuv'], False)

---YVlarv---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
110009328 records with dimensions (66, 52, 94, 341)
110009328 records with dimensions (66, 52, 94, 341)
Runtime: 6.22 minutes
---YVpjuv---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
271689704 records with dimensions (163, 52, 94, 341)
271689704 records with dimensions (163, 52, 94, 341)
Runtime: 6.35 minutes


## Lag indices as needed

For life stages where data from the prior calendar year are relevant for this year's recruitment deviaitons, adjust the index +1. For yellowtail, those life stages are preconditioning, copulation, and egg fertilization.

In the future, it'd be nice to do this programatically by adding the lag to the lifestage dictionary.

In [32]:
# lag computation data by one year
Tcop.index += 1

## Combine and write output

In [33]:
# merge pandas dataframes
output_df = pd.concat([MLDpart, MLDlarv, MLDpjuv, Tcop, Tpart, XVlarv, XVpjuv, YVlarv, YVpjuv], axis = 1)

# sort indices
output_df.sort_index(inplace = True)

# save result
output_df.to_csv("../../Data/MOM6/yellowtail_MOM6.csv")

# preview result
output_df

,MLDpart,MLDlarv,MLDpjuv,Tcop,Tpart,XVlarv,XVpjuv,YVlarv,YVpjuv
year,,,,,,,,,
1993,11.847344,10.908626,6.423645,NaN,9.836757,-0.000999,-0.000844,0.027999,-0.040067
1994,11.592854,11.048667,6.438814,9.196337,9.686905,0.001186,-0.002289,0.024591,-0.079349
1995,10.151123,8.576993,6.172762,9.158390,9.766477,-0.001014,-0.005024,0.037794,-0.077568
1996,8.492238,7.788495,6.754915,8.970429,9.998201,0.003827,-0.003628,0.014691,-0.066082
1997,7.905258,7.302444,5.893259,8.881315,9.266623,0.000995,-0.004295,-0.013485,-0.052476
1998,11.336816,8.476565,6.381606,10.351702,11.196400,0.000302,-0.005756,0.037716,-0.057380
1999,10.057905,8.877838,6.187115,9.564533,9.092877,0.001821,-0.005753,0.063495,-0.083329
2000,8.954042,7.735384,6.148491,8.695372,9.289672,0.000059,-0.003499,0.009336,-0.064703
2001,15.045501,14.069611,6.715825,8.645856,9.066905,0.000034,-0.003023,-0.004522,-0.066351


# Monthly Means

This section was written to look at monthly patterns for the same variables, and to match Megan's format of monthly GLORYS data.

In [34]:
'''
Calculate monthly mean for a 2-D variable

Inputs:
- var_dataset: xarray Dataset with the variable to average
- depth_dataset: xarray Dataset with depth to filter
- ds_var_name: string with the variable name, used to extract from var_dataset
- df_var_name: string with the variable name, used to rename column in returned dataframe, will have depth range appended
- min_lat, max_lat: minimum and maximum latitude (integers or floats, inclusive)
- min_depth, max_depth: minimum and maximum depth to filter the longitudinal extent

Outputs:
- ds_df: pandas dataframe with a filtered monthly average of the variable
'''

def monthly_mean_2d(var_dataset, depth_dataset, ds_var_name, df_var_name, min_lat, max_lat, min_depth, max_depth):
    
    # start timer
    start_time = time.time()
    
    print("---" + df_var_name + " (" + str(min_depth) + "-" + str(max_depth) + ") ---")
    print_ds_info(var_dataset, ds_var_name, nan_bool = False)
    
    # merge files
    ds_merged = xr.merge([var_dataset, depth_dataset])
    print_ds_info(ds_merged, ds_var_name, nan_bool = False)
    
    # select latitudes
    ds_selected = ds_merged.sel(lat = slice(min_lat, max_lat))
    print_ds_info(ds_selected, ds_var_name, nan_bool = False)
    
    # mask depths
    ds_masked = ds_selected.where((ds_selected.deptho >= min_depth) & (ds_selected.deptho <= max_depth))
    print_ds_info(ds_masked, ds_var_name, nan_bool = False)
    
    # calculate mean
    ds_mean = ds_masked[ds_var_name].groupby(['time.year', 'time.month']).mean(dim = ['time', 'lon', 'lat'])
    
    # create pandas dataframe
    ds_df = ds_mean.to_dataframe()
    
    # rename column from dataset to dataframe variable name
    df_var_name_with_depth = df_var_name + "_" + str(min_depth) + "_" + str(max_depth)
    ds_df.rename(columns = {ds_var_name:df_var_name_with_depth}, inplace = True)
    
    # end timer and print runtime
    end_time = time.time()
    print(f'Runtime: {round((end_time - start_time) / 60, 2)} minutes')
    
    # write output
    ds_df.to_csv("../../Data/MOM6/interim/yellowtail_MOM6_monthly_" + df_var_name_with_depth + ".csv")
    
    return ds_df

In [35]:
'''
Calculate monthly mean for a 3-D variable

Inputs:
- var_dataset: xarray Dataset with the variable to average
- depth_dataset: xarray Dataset with depth to filter
- ds_var_name: string with the variable name, used to extract from var_dataset
- df_var_name: string with the variable name, used to rename column in returned dataframe, will have depth range appended
- min_lat, max_lat: minimum and maximum latitude (integers or floats, inclusive)
- min_depth, max_depth: minimum and maximum depth to filter the longitudinal extent

Outputs:
- ds_df: pandas dataframe with a filtered monthly average of the variable
'''

def monthly_mean_3d(var_dataset, depth_dataset, ds_var_name, df_var_name, min_lat, max_lat, min_depth, max_depth):
    
    # start timer
    start_time = time.time()
    
    print("---" + df_var_name + " (" + str(min_depth) + "-" + str(max_depth) + ") ---")
    print_ds_info(var_dataset, ds_var_name, nan_bool = False)
    
    # merge files
    ds_merged = xr.merge([var_dataset, depth_dataset])
    print_ds_info(ds_merged, ds_var_name, nan_bool = False)
    
    # select latitudes
    ds_selected = ds_merged.sel(lat = slice(min_lat, max_lat))
    print_ds_info(ds_selected, ds_var_name, nan_bool = False)
    
    # mask depths
    ds_masked = ds_selected.where((ds_selected.deptho >= min_depth) & (ds_selected.deptho <= max_depth))
    print_ds_info(ds_masked, ds_var_name, nan_bool = False)
    
    # calculate mean
    ds_mean = ds_masked[ds_var_name].groupby(['time.year', 'time.month']).mean(dim = ['time', 'z_l', 'lon', 'lat'])
    
    # create pandas dataframe
    ds_df = ds_mean.to_dataframe()
    
    # rename column from dataset to dataframe variable name
    df_var_name_with_depth = df_var_name + "_" + str(min_depth) + "_" + str(max_depth)
    ds_df.rename(columns = {ds_var_name:df_var_name_with_depth}, inplace = True)
    
    # end timer and print runtime
    end_time = time.time()
    print(f'Runtime: {round((end_time - start_time) / 60, 2)} minutes')
    
    # write output
    ds_df.to_csv("../../Data/MOM6/interim/yellowtail_MOM6_monthly_" + df_var_name_with_depth + ".csv")
    
    return ds_df

It would be more concise code if the 2-d and 3-d functions were the same. The only real difference is the calculate mean step, when `'z_l'` is included as a dimension for a 3-d variable and it is not for a 2-d variable.

## Depth dictionary

Based on [Data Availability](https://docs.google.com/document/d/1P8D0kH2xn4NYBc0ib3rYfSO0KAiyDNQ_kZgG4Qub7qY/edit?tab=t.0#heading=h.z5x77oqxpj4i), define filters bottom depth and latitude.

For northern yellowtail, the bottom depth is used to determine the longitudinal extent, and the average is taken across the whole water column. For example, if the longitudinal extent is bottom depths 90-180m, the netCDF is filtered for isobaths where the depth is \[90, 180], and the average is calculated for all the water \[0, 180] at those depths.

In [36]:
# assign depths to filter, in meters
# filters are inclusive (closed interval), e.g. depth 0-180 includes depths 0 and 180 and every depth between

depth_dict = {
    "pre_cop_egg": { # preconditioning, copulation, and egg fertilization
        "min_depth": 90,
        "max_depth": 180
    }, 
    "part": { # partuition
        "min_depth": 0,
        "max_depth": 180
    },
    "larv": { # larvae
        "min_depth": 0,
        "max_depth": 90
    },
    "pjuv": { # pelagic juvenile
        "min_depth": 30,
        "max_depth": 130
    },
    "bjuv": { # benthic juvenile
        "min_depth": 180,
        "max_depth": 550
    }
}

## Calculate means

In [37]:
# mixed layer depths
MLD_list = [monthly_mean_2d(ds_mld, ds_depth, "MLD_003", "MLD", min_lat = 40, max_lat = 48, min_depth = v['min_depth'], max_depth = v['max_depth']) for v in depth_dict.values()]

---MLD (90-180) ---
110054340 records with dimensions (396, 815, 341)
110054340 records with dimensions (396, 815, 341)
12693384 records with dimensions (396, 94, 341)
12693384 records with dimensions (396, 94, 341)
Runtime: 0.0 minutes
---MLD (0-180) ---
110054340 records with dimensions (396, 815, 341)
110054340 records with dimensions (396, 815, 341)
12693384 records with dimensions (396, 94, 341)
12693384 records with dimensions (396, 94, 341)
Runtime: 0.0 minutes
---MLD (0-90) ---
110054340 records with dimensions (396, 815, 341)
110054340 records with dimensions (396, 815, 341)
12693384 records with dimensions (396, 94, 341)
12693384 records with dimensions (396, 94, 341)
Runtime: 0.0 minutes
---MLD (30-130) ---
110054340 records with dimensions (396, 815, 341)
110054340 records with dimensions (396, 815, 341)
12693384 records with dimensions (396, 94, 341)
12693384 records with dimensions (396, 94, 341)
Runtime: 0.0 minutes
---MLD (180-550) ---
110054340 records with dimensions 

In [38]:
# temperatures
T_list = [monthly_mean_3d(ds_temp, ds_depth, "thetao", "T", min_lat = 40, max_lat = 48, min_depth = v['min_depth'], max_depth = v['max_depth']) for v in depth_dict.values()]

---T (90-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 5.46 minutes
---T (0-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 5.53 minutes
---T (0-90) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 5.6 minutes
---T (30-130) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Ru

In [39]:
# X sea water velocities
XV_list = [monthly_mean_3d(ds_xv, ds_depth, "uo_rotate", "XV", min_lat = 40, max_lat = 48, min_depth = v['min_depth'], max_depth = v['max_depth']) for v in depth_dict.values()]

---XV (90-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.06 minutes
---XV (0-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.55 minutes
---XV (0-90) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.09 minutes
---XV (30-130) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 34

In [40]:
# Y sea water velocities
YV_list = [monthly_mean_3d(ds_yv, ds_depth, "vo_rotate", "YV", min_lat = 40, max_lat = 48, min_depth = v['min_depth'], max_depth = v['max_depth']) for v in depth_dict.values()]

---YV (90-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.51 minutes
---YV (0-180) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.16 minutes
---YV (0-90) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 341)
Runtime: 6.47 minutes
---YV (30-130) ---
5636116200 records with dimensions (390, 52, 815, 341)
5636116200 records with dimensions (390, 52, 815, 341)
650055120 records with dimensions (390, 52, 94, 341)
650055120 records with dimensions (390, 52, 94, 34

## Combine and write output

In [41]:
# merge pandas dataframes for each variable
MLD_monthly = pd.concat(MLD_list, axis = 1)
T_monthly = pd.concat(T_list, axis = 1)
XV_monthly = pd.concat(XV_list, axis = 1)
YV_monthly = pd.concat(YV_list, axis = 1)

# merge pandas dataframes for all variables together
output_monthly_df = pd.concat([MLD_monthly, T_monthly, XV_monthly, YV_monthly], axis = 1)

# save result
output_monthly_df.to_csv("../../Data/MOM6/yellowtail_MOM6_monthly.csv")

# preview result
output_monthly_df.head()

MLD_90_180  MLD_0_180   MLD_0_90  MLD_30_130  MLD_180_550  \
year month                                                              
1993 1       17.902390  15.618670  13.086719   14.110741    30.662983   
     2       14.731369  13.669835  12.492916   12.521150    25.600122   
     3       10.967890  10.188473   9.324335    9.631289    17.669699   
     4        8.829741   7.912402   6.895353    7.549868    13.077017   
     5        6.753925   6.396342   5.999891    6.339486     7.440094   

             T_90_180    T_0_180     T_0_90   T_30_130  T_180_550  XV_90_180  \
year month                                                                     
1993 1       9.733425   9.631277   9.440618   9.624644   9.283408  -0.003129   
     2       9.744449   9.666144   9.519989   9.675626   9.384328  -0.002545   
     3       9.790787   9.742286   9.651759   9.764925   9.319251  -0.000695   
     4      10.222564  10.307321  10.465518  10.392113   9.470942   0.000663   
     5      10.578239  10.809765  11.241906  10.968501   9.667706   0.001135   

            XV_0_180   XV_0_90  XV_30_130  XV_180_550  YV_90_180  YV_0_180  \
year month                                                                   
1993 1     -0.002384 -0.000770  -0.000640   -0.006354   0.084106  0.076134   
     2     -0.002514 -0.002447  -0.002580   -0.004089   0.029135  0.027787   
     3     -0.000333  0.000449   0.001224   -0.002830   0.034548  0.033468   
     4     -0.000311 -0.002420  -0.000742    0.002597   0.015112  0.023468   
     5      0.000631 -0.000460   0.000572    0.000646  -0.021051 -0.015534   

             YV_0_90  YV_30_130  YV_180_550  
year month                                   
1993 1      0.058876   0.068103    0.105916  
     2      0.024868   0.024983    0.034430  
     3      0.031130   0.029565    0.030908  
     4      0.041557   0.026490   -0.006385  
     5     -0.003591  -0.012695   -0.017858